# Part C — MedMamba ablation and cross-validation

Runs after the LOMO notebooks, split over the four Kaggle accounts. Set `PARTC_SPLIT` in the configuration cell to the account's number; nothing else changes.

| `PARTC_SPLIT` | Task | Runs | Question |
|---|---|---|---|
| 1 | ablation | MedMamba without SS2D (→ depthwise conv) × seeds 13/42/2025 @ held-out 400× (3 runs) | Does the result depend on the state-space component? |
| 2 | cv | Phikon @ 40/100/200/400× × folds 0–4 (20 runs) + MedMamba @ 400× fold 0 | Do the LOMO numbers survive a change of test patients? |
| 3 | cv | MedMamba @ 400× folds 1–2 (2 runs) | 〃 |
| 4 | cv | MedMamba @ 400× folds 3–4 (2 runs) | 〃 |

The ablation deltas are measured against the MedMamba ho=400 runs from the LOMO notebook, which use the same seeds, split and training code. A CV fold's patient split depends only on (`CV_K`, fold, `CV_SEED`), so the folds run on different accounts together form one 5-fold CV; the analysis notebook merges them by `run_id`.


In [ ]:
# Part C configuration: MedMamba ablation and patient-level cross-validation.
# Runs after the LOMO notebooks, split over the four Kaggle accounts. Set PARTC_SPLIT to the
# account's number; nothing else changes between accounts.
#   1  ablation  MedMamba without SS2D x seeds 13/42/2025 @ held-out 400x          3 runs
#   2  cv        Phikon @ 40/100/200/400x folds 0-4, MedMamba @ 400x fold 0       21 runs
#   3  cv        MedMamba @ 400x folds 1-2                                          2 runs
#   4  cv        MedMamba @ 400x folds 3-4                                          2 runs
# Leaving PARTC_SPLIT at None runs the whole PARTC_TASK in one session instead.
PARTC_SPLIT = 4
PARTC_TASK = 'ablation'          # used only when PARTC_SPLIT is None: 'ablation' or 'cv'

BREAKHIS_ROOT = None                       # auto-discover under /kaggle/input
BACH_ROOT = None                           # auto-discover; used only by the OOD pass
SOURCE_ROOT = '/kaggle/working/mamba_sources'

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 30
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TASK = 'binary'
NUM_CLASSES = 2

# The POOL the LOMO split draws training data from. Part C iterates its own held-out lists
# (ABLATION_HELD_OUT, CV_PLAN) rather than this one, so every run trains on the other three
# magnifications, exactly as in the LOMO notebooks.
MAGNIFICATIONS = [40, 100, 200, 400]

# --- Ablation ---
# MedMamba's SS_Conv_SSM block splits the channels between a convolution branch and an SS2D
# (2-D selective-scan) branch. medmamba_no_ssm swaps every SS2D for a depthwise-conv block of
# the same width, so its delta measures what the state-space component contributes. This
# variant already built and ran a forward pass on Kaggle (10.91M parameters, 1.65 GFLOPs).
ABLATION_BASE_MODEL = 'medmamba'
ABLATION_VARIANTS = ['medmamba_no_ssm']      # SS2D -> depthwise-conv block
ABLATION_HELD_OUT = 400
ABLATION_SEEDS = [13, 42, 2025]              # the LOMO seeds, hence the LOMO splits
ABLATION_PATIENT_FRACTION = 1.0

# Deltas are measured against the MedMamba ho=400 runs of the LOMO notebook, which share these
# seeds, the split and the training code. Retraining that baseline here costs three more
# MedMamba runs; enable it only if the accounts run different Kaggle environments (the analysis
# notebook compares environment.json across outputs and warns when they differ).
INCLUDE_BASELINE_REFERENCE = False
BASELINE_MODEL = ABLATION_BASE_MODEL
BASELINE_SEEDS = ABLATION_SEEDS

# --- Cross-validation ---
# 5-fold patient-level CV: the frozen Phikon probe at all four held-out magnifications and
# MedMamba at the hardest one. Each entry is (model, held-out magnifications, folds). A fold's
# split depends only on (CV_K, fold, CV_SEED), so folds run on different accounts form one CV.
CV_K = 5
CV_SEED = 13
CV_PATIENT_FRACTION = 1.0
FM_NAME = 'owkin/phikon'
ALL_FOLDS = list(range(CV_K))
CV_PLAN = [(FM_NAME, [40, 100, 200, 400], ALL_FOLDS),
           ('medmamba', [400], ALL_FOLDS)]
CV_HELD_OUT = 400                            # used only by the split preview cell

assert PARTC_SPLIT in (None, 1, 2, 3, 4), f'unknown PARTC_SPLIT {PARTC_SPLIT!r}'
if PARTC_SPLIT == 1:
    PARTC_TASK = 'ablation'
elif PARTC_SPLIT == 2:
    PARTC_TASK = 'cv'
    CV_PLAN = [(FM_NAME, [40, 100, 200, 400], ALL_FOLDS), ('medmamba', [400], [0])]
elif PARTC_SPLIT == 3:
    PARTC_TASK = 'cv'
    CV_PLAN = [('medmamba', [400], [1, 2])]
elif PARTC_SPLIT == 4:
    PARTC_TASK = 'cv'
    CV_PLAN = [('medmamba', [400], [3, 4])]
assert PARTC_TASK in ('ablation', 'cv'), f'unknown PARTC_TASK {PARTC_TASK!r}'

RUN_NAME = f'main_partC_{PARTC_TASK}' + (f'_split{PARTC_SPLIT}' if PARTC_SPLIT else '')
OUTPUT = f'/kaggle/working/{RUN_NAME}'

TEMPERATURE_SCALE = True
MAX_RUNTIME_HOURS = 11.5

# Phikon is loaded only when this session has Phikon folds.
USE_FOUNDATION_MODEL = PARTC_TASK == 'cv' and any(m == FM_NAME for m, _, _ in CV_PLAN)

_n_abl = (len(ABLATION_VARIANTS) * len(ABLATION_SEEDS)
          + (len(BASELINE_SEEDS) if INCLUDE_BASELINE_REFERENCE else 0))
_n_cv = sum(len(mags) * len(folds) for _, mags, folds in CV_PLAN)
print(f'PARTC_SPLIT  = {PARTC_SPLIT}')
print(f'PARTC_TASK   = {PARTC_TASK}')
print(f'RUN_NAME     = {RUN_NAME}')
print(f'OUTPUT       = {OUTPUT}')
if PARTC_TASK == 'ablation':
    print(f'ABLATION     = {ABLATION_BASE_MODEL}: {ABLATION_VARIANTS} x seeds {ABLATION_SEEDS} '
          f'@ ho={ABLATION_HELD_OUT}' + (' (+ baseline reference)' if INCLUDE_BASELINE_REFERENCE else ''))
    print(f'PLANNED RUNS = {_n_abl}')
else:
    for _m, _mags, _folds in CV_PLAN:
        print(f'CV           = {CV_K}-fold {_m} @ ho={_mags} folds {_folds}')
    print(f'PLANNED RUNS = {_n_cv}')

# Names the verbatim Part B cells still reference.
PILOT_HELD_OUT = ABLATION_HELD_OUT
PILOT_SEEDS = [ABLATION_SEEDS[0]]
MC_DROPOUT_PASSES = 20
OOD_METHODS = []
# The LOMO notebooks write the complete compute report, so Part C builds nothing extra for it.
FROM_SCRATCH_MODELS = []
CHECKPOINT_RULE = 'maximum validation macro-F1'


In [ ]:
# Environment. Install only what is missing; do not downgrade Kaggle-provided packages.
import os, sys, json, random, time, hashlib, platform, subprocess, shutil, re
from pathlib import Path
from datetime import datetime, timezone

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
Path(OUTPUT).mkdir(parents=True, exist_ok=True)
Path(SOURCE_ROOT).mkdir(parents=True, exist_ok=True)

def pip_install_if_missing(module_name, spec=None):
    try:
        __import__(module_name); return True
    except ImportError:
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', spec or module_name], check=True)
            return True
        except subprocess.CalledProcessError as exc:
            print(f'Install failed for {spec or module_name}:', exc); return False

for module_name, spec in [
    ('timm', 'timm'),
    ('einops', 'einops'),
    ('transformers', 'transformers'),
    ('huggingface_hub', 'huggingface_hub'),
    ('fvcore', 'fvcore'),
]:
    pip_install_if_missing(module_name, spec)

# Ensure a timm version new enough for MambaVision (timm >= 1.0 exposes mambavision_tiny_1k).
import timm
if int(timm.__version__.split('.')[0]) < 1:
    print(f'timm {timm.__version__} may be too old for MambaVision; upgrading...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'timm'], check=True)
    import importlib; timm = importlib.reload(timm)
MAMBAVISION_OK = 'mambavision_tiny_1k' in timm.list_models('mambavision*')
print(f'timm = {timm.__version__}    MambaVision available: {MAMBAVISION_OK}')

# Optional compiled Mamba (for MedMamba/DSA-Mamba): try latest, then older pins; skip on failure.
MAMBA_SSM_OK = False
try:
    import mamba_ssm; MAMBA_SSM_OK = True
    print('mamba_ssm already installed:', mamba_ssm.__version__)
except ImportError:
    for spec in [
        ['mamba-ssm', 'causal-conv1d'],
        ['mamba-ssm==2.2.4', 'causal-conv1d==1.5.0.post8'],
        ['mamba-ssm==2.2.2', 'causal-conv1d==1.4.0'],
        ['mamba-ssm==1.2.0.post1', 'causal-conv1d==1.2.0.post2'],
    ]:
        try:
            print('Trying compiled mamba_ssm install:', spec)
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation'] + spec, check=True)
            import importlib; mamba_ssm = importlib.import_module('mamba_ssm'); MAMBA_SSM_OK = True
            print(' -> loaded mamba_ssm:', mamba_ssm.__version__); break
        except Exception as exc:
            print(' -> failed:', str(exc)[:180])
    if not MAMBA_SSM_OK:
        print('mamba_ssm unavailable; MedMamba and DSA-Mamba will be skipped. MambaVision through timm still provides the Mamba axis if MAMBAVISION_OK.')

import numpy as np, pandas as pd, torch, torch.nn as nn
import torchvision, sklearn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, brier_score_loss)

assert torch.cuda.is_available(), 'Kaggle GPU accelerator is required.'
DEVICE = torch.device('cuda:0')

def command_text(args):
    try: return subprocess.check_output(args, text=True, stderr=subprocess.STDOUT).strip()
    except Exception as exc: return str(exc)

env = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version, 'platform': platform.platform(),
    'torch': torch.__version__, 'torch_cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(),
    'gpu_name': torch.cuda.get_device_name(0),
    'gpu_capability': torch.cuda.get_device_capability(0),
    'timm': timm.__version__, 'sklearn': sklearn.__version__,
    'mambavision_ok': MAMBAVISION_OK, 'mamba_ssm_ok': MAMBA_SSM_OK,
    'pip_freeze': command_text([sys.executable, '-m', 'pip', 'freeze']),
    'nvidia_smi': command_text(['nvidia-smi']),
}
Path(OUTPUT, 'environment.json').write_text(json.dumps(env, indent=2, default=str))
config = {k: v for k, v in globals().items()
          if k.isupper() and k not in {'OUTPUT', 'SOURCE_ROOT', 'DEVICE', 'BREAKHIS_ROOT', 'BACH_ROOT'}}
Path(OUTPUT, 'config.json').write_text(json.dumps(config, indent=2, default=str))
print(json.dumps({k: env[k] for k in ('gpu_name','torch','torch_cuda','cudnn','timm','mambavision_ok','mamba_ssm_ok')}, indent=2))


In [ ]:
# Optional MedMamba + DSA-Mamba source pin (only if mamba_ssm compiled). Each Mamba variant is independent.
EXTERNAL_FACTORIES = {}
source_manifest = {}

def clone_at(name, url, commit=None, force_refresh=False):
    destination = Path(SOURCE_ROOT, name)
    if destination.exists() and force_refresh: shutil.rmtree(destination)
    if not destination.exists():
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'fetch', '--all', '--tags'], check=True)
    if commit is not None:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    return destination

if MAMBA_SSM_OK:
    # --- MedMamba (pinned commit) ---
    try:
        med_src = clone_at('MedMamba', 'https://github.com/YubiaoYue/MedMamba.git',
                           commit='104beb3d641d67cef6f09b94ac299083d91bc417')
        safe_med = Path(SOURCE_ROOT, 'medmamba_safe.py')
        source = Path(med_src, 'MedMamba.py').read_text()
        marker = '\nmedmamba_t = VSSM'
        safe_med.write_text(source.split(marker, 1)[0] + '\n' if marker in source else source)
        sys.path.insert(0, str(Path(SOURCE_ROOT)))
        import importlib.util
        spec = importlib.util.spec_from_file_location('medmamba_safe', safe_med)
        med_module = importlib.util.module_from_spec(spec); spec.loader.exec_module(med_module)
        MedMambaVSSM = med_module.VSSM
        EXTERNAL_FACTORIES['medmamba'] = lambda n: MedMambaVSSM(depths=[2,2,4,2], dims=[96,192,384,768], num_classes=n)
        source_manifest['MedMamba_commit'] = command_text(['git','-C',str(med_src),'rev-parse','HEAD'])
        print('MedMamba registered.')
    except Exception as exc:
        print('MedMamba registration failed:', repr(exc)[:180])

    # --- DSA-Mamba (try pinned, fall back to HEAD) ---
    for attempt_commit in ['7af166288e5e0b710481788c15af79326017b37b', None]:
        try:
            dsa_src = clone_at('DSA-Mamba', 'https://github.com/First-Ronin/DSA-Mamba.git',
                               commit=attempt_commit, force_refresh=(attempt_commit is None))
            sys.path.insert(0, str(dsa_src))
            for mod in list(sys.modules):
                if mod.startswith('model'): del sys.modules[mod]
            from model.DSAmamba import VSSM as DSAMambaVSSM
            EXTERNAL_FACTORIES['dsamamba'] = lambda n: DSAMambaVSSM(in_chans=3, num_classes=n)
            source_manifest['DSA_Mamba_commit'] = command_text(['git','-C',str(dsa_src),'rev-parse','HEAD'])
            source_manifest['DSA_Mamba_attempt'] = 'pinned' if attempt_commit else 'HEAD_fallback'
            print(f'DSA-Mamba registered ({source_manifest["DSA_Mamba_attempt"]}).'); break
        except Exception as exc:
            print(f'DSA-Mamba attempt ({attempt_commit or "HEAD"}) failed:', repr(exc)[:180])
else:
    print('Skipping MedMamba + DSA-Mamba (mamba_ssm not available). MambaVision through timm supplies the Mamba axis.')

if source_manifest:
    Path(OUTPUT, 'source_manifest.json').write_text(json.dumps(source_manifest, indent=2))

# Append registered external Mamba variants to FROM_SCRATCH_MODELS.
for name in ('medmamba', 'dsamamba'):
    if name in EXTERNAL_FACTORIES and name not in FROM_SCRATCH_MODELS:
        FROM_SCRATCH_MODELS.append(name)

# Remove MambaVision from the active list if the installed timm can't build it.
if not MAMBAVISION_OK and 'mambavision_tiny_1k' in FROM_SCRATCH_MODELS:
    FROM_SCRATCH_MODELS.remove('mambavision_tiny_1k')
    print('Removed mambavision_tiny_1k from active models (not in installed timm).')

print('\nActive from-scratch models:', FROM_SCRATCH_MODELS)


In [ ]:
# ---------- Optional: restore DSA-Mamba's missing cross_attention.py ----------
# Part A's log shows DSA-Mamba failed to register on both the pinned commit and HEAD:
#     ModuleNotFoundError: No module named 'model.cross_attention'
# Cause: model/DSAmamba.py line 12 does `from .cross_attention import CrossAttention`,
# but the upstream repo no longer ships that file. Its git history explains why:
#     2025-06-09  First-Ronin  94c5042  "Add files via upload"     <- file present
#     2025-07-30  First-Ronin           "Delete model directory"   <- model/ removed
#   DSAmamba.py was later re-added; cross_attention.py never was. Verified directly:
#   the 94c5042 raw URL below serves the file, while the current upstream model/
#   tree contains only DSAmamba.py. (Commit SHAs beyond 94c5042 are from a
#   summarized listing and are deliberately not cited as fact.)
# The only fork (junaidmaqbool) has a cross_attention.py, but its history shows six
# commits rewriting it (LazyLinear, 4-D support, device fixes), so that copy is NOT the
# published module. We therefore restore the ORIGINAL file from the author's own commit
# 94c5042 -- recovering deleted upstream code, not reconstructing it.
#
# This cell does NOT add DSA-Mamba to any training queue. It only tries to build the
# model and run one forward pass, so you can decide from evidence whether including
# DSA-Mamba is viable. Two known risks:
#   1. Version pairing -- cross_attention.py is from 2025-06-09 while DSAmamba.py is
#      pinned to a later commit; the author never had both in the tree simultaneously.
#   2. Memory -- Cross_Attention computes `value.transpose(1,2) @ key`, which is
#      (B, HW, HW). At stage 0 of a 224x224 input HW = 56*56 = 3136, so one head costs
#      B * 3136^2 * 4 bytes (~1.26 GB at B=32). The smoke test below uses B=2 and
#      reports peak memory so you can extrapolate.
TRY_DSAMAMBA_RESTORE = False   # DSA-Mamba is not part of this study
DSAMAMBA_CROSS_ATTENTION_COMMIT = '94c5042'      # First-Ronin, 2025-06-09

if not TRY_DSAMAMBA_RESTORE:
    print('DSA-Mamba restore skipped (TRY_DSAMAMBA_RESTORE = False).')
elif 'dsamamba' in EXTERNAL_FACTORIES:
    print('DSA-Mamba already registered by the previous cell; nothing to restore.')
elif not MAMBA_SSM_OK:
    print('DSA-Mamba restore skipped: mamba_ssm unavailable, so DSAmamba cannot run.')
else:
    import urllib.request, importlib, torch
    DSA_DIR = Path(SOURCE_ROOT, 'DSA-Mamba')
    target = Path(DSA_DIR, 'model', 'cross_attention.py')
    url = ('https://raw.githubusercontent.com/First-Ronin/DSA-Mamba/'
           f'{DSAMAMBA_CROSS_ATTENTION_COMMIT}/model/cross_attention.py')
    try:
        if not DSA_DIR.exists():
            raise RuntimeError(f'{DSA_DIR} not found - the clone in the previous cell '
                               f'must succeed before the file can be restored.')
        if target.exists():
            print(f'cross_attention.py already present at {target}')
        else:
            print(f'Fetching original cross_attention.py from commit '
                  f'{DSAMAMBA_CROSS_ATTENTION_COMMIT} ...')
            with urllib.request.urlopen(url, timeout=60) as resp:
                blob = resp.read()
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(blob)
            print(f'  wrote {target}  ({len(blob)} bytes)')
            # Record provenance next to the file so the origin is never ambiguous later.
            Path(DSA_DIR, 'model', 'cross_attention.PROVENANCE.txt').write_text(
                f'Restored from {url}\n'
                f'Author: First-Ronin (original repo owner)\n'
                f'Commit: {DSAMAMBA_CROSS_ATTENTION_COMMIT} "Add files via upload", 2025-06-09\n'
                f'Reason: a later upstream commit deleted the model directory. DSAmamba.py\n'
                f'        was re-added afterwards, this file was not, yet DSAmamba.py line 12\n'
                f'        still does `from .cross_attention import CrossAttention`.\n'
                f'Verified: the URL above serves this file, and the current upstream model/\n'
                f'          tree contains only DSAmamba.py.\n'
                f'NOT the junaidmaqbool fork version, which was rewritten in 6 commits.\n',
                encoding='utf-8')

        sanity = target.read_text(encoding='utf-8', errors='replace')
        assert 'class CrossAttention' in sanity, \
            'restored file does not define CrossAttention - refusing to use it'
        print(f'  defines: ' + ', '.join(
            sorted({ln.split("(")[0].replace("class ", "").strip()
                    for ln in sanity.splitlines() if ln.startswith("class ")})))

        # Re-import cleanly: drop any half-initialised `model.*` modules from the failed
        # attempt in the previous cell, then import again.
        if str(DSA_DIR) not in sys.path:
            sys.path.insert(0, str(DSA_DIR))
        for mod in [m for m in list(sys.modules) if m == 'model' or m.startswith('model.')]:
            del sys.modules[mod]
        importlib.invalidate_caches()
        from model.DSAmamba import VSSM as DSAMambaVSSM
        print('  import model.DSAmamba OK')

        # Smoke test: build + one forward pass at B=2, reporting peak memory so the
        # B=32 training cost can be extrapolated before committing GPU hours.
        _dev = DEVICE
        m = DSAMambaVSSM(in_chans=3, num_classes=NUM_CLASSES).to(_dev).eval()
        n_params = sum(p.numel() for p in m.parameters())
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            out = m(torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=_dev))
        assert tuple(out.shape) == (2, NUM_CLASSES), f'unexpected output {tuple(out.shape)}'
        peak_mib = (torch.cuda.max_memory_allocated() / 2**20) if torch.cuda.is_available() else float('nan')
        print(f'  FORWARD OK  params={n_params/1e6:.2f}M  output={tuple(out.shape)}')
        if peak_mib == peak_mib:
            print(f'  peak GPU memory at B=2: {peak_mib:.0f} MiB'
                  f'  -> rough estimate at B={BATCH_SIZE}: {peak_mib * BATCH_SIZE / 2:.0f} MiB')
            if peak_mib * BATCH_SIZE / 2 > 13000:
                print(f'  WARNING: extrapolated training memory exceeds a T4 (~15 GiB usable).')
                print(f'           Consider a smaller BATCH_SIZE for DSA-Mamba specifically.')
        del m
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        EXTERNAL_FACTORIES['dsamamba'] = lambda n: DSAMambaVSSM(in_chans=3, num_classes=n)
        source_manifest['DSA_Mamba_cross_attention_restored_from'] = \
            f'First-Ronin@{DSAMAMBA_CROSS_ATTENTION_COMMIT}'
        try:
            source_manifest['DSA_Mamba_commit'] = command_text(
                ['git', '-C', str(DSA_DIR), 'rev-parse', 'HEAD']).strip()
        except Exception:
            pass
        Path(OUTPUT, 'source_manifest.json').write_text(json.dumps(source_manifest, indent=2))
        print('\nDSA-Mamba REGISTERED and verified.')
        print('  It is NOT in Part C\'s training queue. To evaluate it you need LOMO runs at')
        print('  every magnification, which belongs in a separate session - see the notes below.')
    except Exception as exc:
        print(f'\nDSA-Mamba restore FAILED: {repr(exc)[:400]}')
        print('  Report this as a reproducibility limitation rather than working around it:')
        print('  the published repo is missing model/cross_attention.py, which its core')
        print('  SS_Conv_SSM block imports, so DSA-Mamba cannot be built as published.')


In [ ]:
# BreakHis manifest with patient-disjoint LOMO split (properly stratified per class).
# Slide ids in BreakHis are alphanumeric (22549AB, 23060AB, 16184CD, 3411F), not
# purely numeric. The earlier pattern required -\d+-\d+ here, so build_manifest's
# `if not match: continue` silently discarded every suffixed slide: 7909 files on
# disk became 5792 rows, 82 patients became 59, and because benign slides carry a
# letter suffix far more often, 17 of the 24 benign patients were lost.
NAME = re.compile(r'^(?P<patient>SOB_(?P<coarse>[BM])_(?P<subtype>[A-Z]+)-\d+-[A-Za-z0-9]+)-(?P<magnification>\d+)-\d+', re.I)
SUBTYPE_TO_ID = {'A': 0, 'F': 1, 'PT': 2, 'TA': 3, 'DC': 4, 'LC': 5, 'MC': 6, 'PC': 7}

def discover_breakhis_root(hint=None, search_root='/kaggle/input'):
    candidates = []
    if hint and Path(hint).exists():
        candidates.append(Path(hint))
    if Path(search_root).exists():
        for top in sorted(Path(search_root).iterdir()):
            if top.is_dir():
                candidates.append(top)
        candidates.append(Path(search_root))
    best = None
    best_count = 0
    for candidate in candidates:
        try:
            count = sum(1 for _ in Path(candidate).rglob('SOB_*.png'))
        except (PermissionError, OSError):
            continue
        if count > best_count:
            best = candidate
            best_count = count
    if best is None or best_count == 0:
        raise RuntimeError('Could not auto-discover BreakHis root. Attach ambarish/breakhis or set BREAKHIS_ROOT.')
    print('Auto-discovered BREAKHIS_ROOT =', best, ' (', best_count, 'SOB PNG files)')
    return str(best)

if BREAKHIS_ROOT is None:
    BREAKHIS_ROOT = discover_breakhis_root()
elif not list(Path(BREAKHIS_ROOT).rglob('SOB_*.png')):
    print('Supplied BREAKHIS_ROOT=', BREAKHIS_ROOT, 'has no SOB files; retrying auto-discovery.')
    BREAKHIS_ROOT = discover_breakhis_root()

def build_manifest(root):
    rows = []
    for path in sorted(Path(root).rglob('SOB_*.png')):
        match = NAME.match(path.name)
        if not match:
            continue
        item = match.groupdict()
        subtype = item['subtype'].upper()
        if subtype not in SUBTYPE_TO_ID:
            continue
        rows.append({
            'path': str(path),
            'patient_id': item['patient'],
            'magnification': int(item['magnification']),
            'coarse_label': int(item['coarse'].upper() == 'M'),
            'subtype': subtype,
            'subtype_label': SUBTYPE_TO_ID[subtype],
        })
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError('No matching images found under ' + str(root))
    return frame

def patient_lomo_split(frame, held_out_mag, fraction=1.0, seed=2025,
                       val_ratio=0.15, test_ratio=0.20, val_on_held_out=False):
    """Patient-disjoint LOMO split with PROPER per-class stratification.
    Both benign and malignant patients are proportionally represented in every split
    (train, val, test) to prevent single-class val/test collapse.
    """
    frame = frame.copy()
    patient_class = frame.groupby('patient_id').coarse_label.first().reset_index()
    rng = np.random.default_rng(seed)
    train_patients, val_patients, test_patients = set(), set(), set()
    for cls in sorted(patient_class.coarse_label.unique()):
        group = patient_class.loc[patient_class.coarse_label == cls, 'patient_id'].to_numpy()
        perm = rng.permutation(group)
        n = len(perm)
        n_test = max(1, int(round(test_ratio * n)))
        n_val = max(1, int(round(val_ratio * n)))
        assert n_test + n_val < n, 'Class ' + str(cls) + ' has only ' + str(n) + ' patients; val+test ratios too high.'
        test_patients.update(perm[:n_test].tolist())
        val_patients.update(perm[n_test:n_test + n_val].tolist())
        remain = perm[n_test + n_val:].tolist()
        n_keep = max(1, int(round(fraction * len(remain))))
        train_patients.update(remain[:n_keep])
    train_mags = [m for m in MAGNIFICATIONS if m != held_out_mag]
    frame['split'] = 'drop'
    frame.loc[frame.patient_id.isin(train_patients) & frame.magnification.isin(train_mags), 'split'] = 'train'
    if val_on_held_out:
        frame.loc[frame.patient_id.isin(val_patients) & (frame.magnification == held_out_mag), 'split'] = 'val'
    else:
        frame.loc[frame.patient_id.isin(val_patients) & frame.magnification.isin(train_mags), 'split'] = 'val'
    frame.loc[frame.patient_id.isin(test_patients) & (frame.magnification == held_out_mag), 'split'] = 'test'
    subset = frame.loc[frame.split != 'drop'].copy()
    leak = subset.groupby('patient_id').split.nunique().gt(1).any()
    assert not leak, 'Patient leakage detected.'
    for name in ('train', 'val', 'test'):
        rows = subset[subset.split == name]
        if len(rows) == 0:
            raise RuntimeError('Split ' + name + ' empty (held_out=' + str(held_out_mag) + ', fraction=' + str(fraction) + ', seed=' + str(seed) + ').')
        counts = rows.coarse_label.value_counts().to_dict()
        if len(counts) < 2:
            raise RuntimeError('Split ' + name + ' has only one class ' + str(counts) + ' -- stratification failed.')
    return subset

MANIFEST = build_manifest(BREAKHIS_ROOT)
MANIFEST.to_csv(Path(OUTPUT, 'breakhis_manifest.csv'), index=False)
manifest_hash = hashlib.sha256(MANIFEST.to_csv(index=False).encode()).hexdigest()
Path(OUTPUT, 'breakhis_manifest.sha256').write_text(manifest_hash + chr(10))
print('total images:', len(MANIFEST), '  patients:', MANIFEST.patient_id.nunique())
print('split-manifest SHA-256:', manifest_hash)
print(MANIFEST.groupby(['magnification', 'coarse_label']).size().unstack(fill_value=0))
print('per-patient class counts:', MANIFEST.groupby('patient_id').coarse_label.first().value_counts().to_dict())

_probe = patient_lomo_split(MANIFEST, PILOT_HELD_OUT, fraction=1.0, seed=PILOT_SEEDS[0])
print('Split sanity check at held_out_mag=' + str(PILOT_HELD_OUT) + ':')
print(_probe.groupby(['split', 'magnification']).size().unstack(fill_value=0))
print('patients per split:', _probe.groupby('split').patient_id.nunique().to_dict())
print('class balance per split:')
for name in ('train', 'val', 'test'):
    counts = _probe[_probe.split == name].coarse_label.value_counts().to_dict()
    print('  ' + name + ':', counts)

class HistologyDataset(Dataset):
    def __init__(self, rows, train, mean=(.485, .456, .406), std=(.229, .224, .225)):
        self.rows = rows.reset_index(drop=True)
        ops = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
        if train:
            ops += [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                    transforms.RandomRotation(15), transforms.ColorJitter(.1, .1, .05)]
        self.transform = transforms.Compose(ops + [transforms.ToTensor(),
                                                    transforms.Normalize(mean, std)])
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows.iloc[index]
        y = int(row.coarse_label if TASK == 'binary' else row.subtype_label)
        return self.transform(Image.open(row.path).convert('RGB')), y, row.patient_id


In [ ]:
# ---------- Resume from a previous Kaggle version of this notebook ----------
# Every "Save & Run All" commit starts with an EMPTY /kaggle/working, so the protocol
# cell's record.json skip cannot see runs finished by an earlier commit on its own. A
# commit that stops at the time budget is continued like this:
#   1. Add Input -> Your Work -> this notebook -> the version that stopped
#   2. Save & Run All again
# This cell finds that version's `<RUN_NAME>/runs` folder under /kaggle/input and copies
# the finished runs into OUTPUT before the protocol cell builds its queue.
#
# Runs are copied ONLY if they were trained on exactly this manifest. Outputs from before
# the manifest-regex fix (5792 images / 59 patients) must never leak into a corrected run,
# so a fingerprint of the attached breakhis_manifest.csv is compared with this session's
# MANIFEST first. The fingerprint uses file names rather than absolute paths, because the
# mount point under /kaggle/input can differ between sessions.
import hashlib, shutil
RESUME_FROM_INPUT = True

def _manifest_fingerprint(frame):
    view = pd.DataFrame({
        'file': frame['path'].map(lambda p: Path(str(p)).name),
        'patient_id': frame['patient_id'].astype(str),
        'magnification': frame['magnification'].astype(int),
        'coarse_label': frame['coarse_label'].astype(int),
    }).sort_values(['file', 'magnification', 'patient_id']).reset_index(drop=True)
    return hashlib.sha256(view.to_csv(index=False).encode()).hexdigest()

def _find_previous_outputs(run_name, search_root='/kaggle/input'):
    root = Path(search_root)
    if not root.exists():
        return []
    return sorted({cand for cand in root.rglob(run_name)
                   if cand.is_dir() and (cand / 'runs').is_dir()})

if not RESUME_FROM_INPUT:
    print('Resume disabled (RESUME_FROM_INPUT = False).')
else:
    _here = _manifest_fingerprint(MANIFEST)
    _prev = _find_previous_outputs(RUN_NAME)
    if not _prev:
        print(f'No previous output named {RUN_NAME!r} under /kaggle/input - starting fresh.')
    _resumed = 0
    for _src in _prev:
        _man = _src / 'breakhis_manifest.csv'
        if not _man.exists():
            print(f'  SKIPPED {_src}: no breakhis_manifest.csv, so its runs cannot be verified.')
            continue
        _theirs = pd.read_csv(_man)
        if _manifest_fingerprint(_theirs) != _here:
            print(f'  REFUSED {_src}')
            print(f'    its runs were trained on a different manifest '
                  f'({len(_theirs)} rows, {_theirs.patient_id.nunique()} patients) than this '
                  f'session ({len(MANIFEST)} rows, {MANIFEST.patient_id.nunique()} patients).')
            print('    Most likely an output from before the manifest fix; those runs are retrained.')
            continue
        _n = 0
        # record.json is written only when a run finishes, so an interrupted run (best.pt
        # but no record.json) is not copied and gets retrained.
        for _rec in sorted((_src / 'runs').glob('*/record.json')):
            _dst = Path(OUTPUT, 'runs', _rec.parent.name)
            if (_dst / 'record.json').exists():
                continue
            shutil.copytree(_rec.parent, _dst, dirs_exist_ok=True)
            _n += 1
        _resumed += _n
        print(f'  resumed {_n} finished runs from {_src}')
    if _prev:
        print(f'Total resumed: {_resumed}. The protocol cell skips these and rebuilds raw_runs.csv.')


In [ ]:
# Model factory + parameter/FLOP report.
# NOTE: parameter counting fixed here relative to Part A/B, which summed every
# entry of fvcore's prefix-keyed parameter_count() and so inflated all counts
# by roughly the module-tree depth (~7.7x in Part A's from_scratch_compute_report.csv).
def build_from_scratch(name, num_classes):
    if name == 'convnext_tiny':
        return timm.create_model('convnext_tiny', pretrained=True, num_classes=num_classes)
    if name == 'swin_tiny':
        return timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=num_classes)
    if name == 'mambavision_tiny_1k':
        # Pretrained weights come from Hugging Face; requires Internet on the first call.
        return timm.create_model('mambavision_tiny_1k', pretrained=True, num_classes=num_classes)
    if name in EXTERNAL_FACTORIES:
        return EXTERNAL_FACTORIES[name](num_classes)
    raise KeyError(f'Unknown or unconfigured model: {name}')

def parameter_and_flop_report(name, model):
    try:
        from fvcore.nn import FlopCountAnalysis, parameter_count
        model.eval().to(DEVICE)
        dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
        try: flops = FlopCountAnalysis(model, dummy).total()
        except Exception as exc: flops = float('nan'); print(f'FLOP count skipped for {name}:', str(exc)[:120])
        # parameter_count() is keyed by module prefix and each entry already includes
        # its submodules, so summing every value counts each parameter once per tree
        # level. The root key '' is the true total.
        pc = parameter_count(model)
        params = int(pc.get('', 0)) or sum(p.numel() for p in model.parameters())
        direct = sum(p.numel() for p in model.parameters())
        if params != direct:
            print(f'  {name}: parameter_count root={params:,} vs numel sum={direct:,} '
                  f'-- using numel sum')
            params = direct
    except Exception as exc:
        params = sum(p.numel() for p in model.parameters()); flops = float('nan')
        print(f'fvcore unavailable for {name}:', str(exc)[:120])
    return {'model': name, 'parameters': int(params),
            'flops_g_at_224': flops / 1e9 if flops == flops else float('nan')}

report = []
for name in list(FROM_SCRATCH_MODELS):
    try:
        m = build_from_scratch(name, NUM_CLASSES)
        report.append(parameter_and_flop_report(name, m))
        del m; torch.cuda.empty_cache()
    except Exception as exc:
        print(f'Build failed for {name}, removing from active list:', repr(exc)[:200])
        if name in FROM_SCRATCH_MODELS: FROM_SCRATCH_MODELS.remove(name)
pd.DataFrame(report).to_csv(Path(OUTPUT, 'from_scratch_compute_report.csv'), index=False)
print('\nActive from-scratch models after build check:', FROM_SCRATCH_MODELS)

# ---- Record the actual initialization regime, which the branch name does not convey ----
# build_from_scratch() uses pretrained=True for the timm encoders, so ConvNeXt-T and
# Swin-T start from ImageNet-1k weights, while MedMamba is built through
# EXTERNAL_FACTORIES with no checkpoint and is therefore randomly initialised. All of
# them are nevertheless recorded as branch='from_scratch'. Part C keeps this behaviour
# unchanged so its baseline stays comparable with Part A/B, but the log should say so.
print('\nInitialization regime actually used by build_from_scratch():')
_INIT_NOTE = {
    'convnext_tiny':       'ImageNet-1k pretrained, fully fine-tuned  (pretrained=True)',
    'swin_tiny':           'ImageNet-1k pretrained, fully fine-tuned  (pretrained=True)',
    'mambavision_tiny_1k': 'ImageNet pretrained via HF, fully fine-tuned  (pretrained=True)',
}
for _n in FROM_SCRATCH_MODELS:
    print(f'  {_n:<22} {_INIT_NOTE.get(_n, "RANDOM INIT - genuinely trained from scratch")}')
print('  NOTE: all of the above are recorded as branch="from_scratch". Only the randomly')
print('        initialised models are from scratch in the literal sense; describe the')
print('        comparison accordingly when writing it up.')
pd.DataFrame(report)


In [ ]:
# Copied verbatim from the LOMO notebook (Part B cell-7) so the CV task probes Phikon
# with exactly the extractor and probe head used for the LOMO results. It loads the
# model only when USE_FOUNDATION_MODEL is True, i.e. in the CV task.
# Frozen foundation-model feature extractor. Phikon is public.
FM_MODEL = None; FM_PROCESSOR = None; FM_DIM = None

def build_foundation_extractor(name):
    from transformers import AutoModel, AutoImageProcessor
    token = os.environ.get('HF_TOKEN', None)
    try:
        processor = AutoImageProcessor.from_pretrained(name, token=token, trust_remote_code=True)
    except Exception:
        processor = None
    model = AutoModel.from_pretrained(name, token=token, trust_remote_code=True).eval().to(DEVICE)
    for parameter in model.parameters(): parameter.requires_grad_(False)
    return model, processor

class LinearProbe(nn.Module):
    def __init__(self, feature_dim, num_classes):
        super().__init__(); self.head = nn.Linear(feature_dim, num_classes)
    def forward(self, x): return self.head(x)

@torch.inference_mode()
def extract_features(model, loader):
    feats, labels, patients = [], [], []
    for x, y, pid in loader:
        x = x.to(DEVICE, non_blocking=True)
        try: out = model(pixel_values=x)
        except TypeError: out = model(x)
        if hasattr(out, 'last_hidden_state'):
            token = out.last_hidden_state[:, 0]
        elif hasattr(out, 'pooler_output') and out.pooler_output is not None:
            token = out.pooler_output
        else:
            token = out if out.dim() == 2 else out.mean(dim=list(range(1, out.dim() - 1)))
        feats.append(token.detach().cpu()); labels.append(torch.as_tensor(y)); patients.extend(pid)
    return torch.cat(feats), torch.cat(labels), patients

if USE_FOUNDATION_MODEL:
    try:
        FM_MODEL, FM_PROCESSOR = build_foundation_extractor(FM_NAME)
        with torch.inference_mode():
            dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
            try: out = FM_MODEL(pixel_values=dummy)
            except TypeError: out = FM_MODEL(dummy)
            if hasattr(out, 'last_hidden_state'):
                FM_DIM = out.last_hidden_state.shape[-1]
            elif hasattr(out, 'pooler_output') and out.pooler_output is not None:
                FM_DIM = out.pooler_output.shape[-1]
            else:
                FM_DIM = out.shape[-1] if out.dim() == 2 else out.mean(dim=list(range(1, out.dim() - 1))).shape[-1]
        print(f'Foundation model {FM_NAME} loaded; feature dim = {FM_DIM}')
    except Exception as exc:
        print(f'Foundation model load failed: {repr(exc)[:200]}. Continuing without the FM branch.')
        USE_FOUNDATION_MODEL = False


In [ ]:
# Calibration and selective-prediction utilities.
# numpy 2.x renamed trapz -> trapezoid and deprecated the old name, which otherwise
# emits a DeprecationWarning on every metric computation and buries the run log.
# The two are numerically identical, so this alias changes no results.
_trapz = getattr(np, 'trapezoid', None) or np.trapz

def expected_calibration_error(probs, labels, n_bins=15):
    confidences = probs.max(axis=1); predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for lower, upper in zip(bins[:-1], bins[1:]):
        in_bin = (confidences > lower) & (confidences <= upper)
        if in_bin.sum() == 0: continue
        gap = abs(accuracies[in_bin].mean() - confidences[in_bin].mean())
        ece += in_bin.mean() * gap
    return float(ece)

def brier_multiclass(probs, labels):
    one_hot = np.eye(probs.shape[1])[labels]
    return float(((probs - one_hot) ** 2).sum(axis=1).mean())

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__(); self.log_temp = nn.Parameter(torch.zeros(1))
    def forward(self, logits): return logits / torch.exp(self.log_temp)

def fit_temperature(val_logits, val_labels, max_iter=200):
    scaler = TemperatureScaler().to(DEVICE)
    logits = torch.as_tensor(val_logits, device=DEVICE, dtype=torch.float32)
    labels = torch.as_tensor(val_labels, device=DEVICE, dtype=torch.long)
    optim = torch.optim.LBFGS([scaler.log_temp], lr=0.1, max_iter=max_iter)
    loss_fn = nn.CrossEntropyLoss()
    def closure():
        optim.zero_grad(); loss = loss_fn(scaler(logits), labels); loss.backward(); return loss
    optim.step(closure)
    return float(torch.exp(scaler.log_temp).item())

def risk_coverage_curve(probs, labels):
    confidences = probs.max(axis=1); predictions = probs.argmax(axis=1)
    order = np.argsort(-confidences)
    errors = (predictions != labels).astype(float)[order]
    coverage = np.arange(1, len(order) + 1) / len(order)
    risk = np.cumsum(errors) / np.arange(1, len(order) + 1)
    return coverage, risk, float(_trapz(risk, coverage))

def maximum_softmax_probability(probs): return probs.max(axis=1)
def energy_score(logits, T=1.0):
    logits = np.asarray(logits, dtype=np.float64)
    return T * (np.log(np.exp(logits / T).sum(axis=1) + 1e-12))

print('Calibration and selective-prediction utilities ready.')


In [ ]:
# Training + evaluation primitives.
STANDARD_RECORD_COLUMNS = ['run_id', 'branch', 'model', 'held_out_mag', 'patient_fraction', 'seed',
                            'peak_memory_mib', 'wallclock_s',
                            'test_accuracy', 'test_balanced_accuracy', 'test_macro_f1',
                            'test_ece', 'test_brier', 'test_auc',
                            'test_risk_coverage_auc', 'test_risk_at_coverage_80',
                            'test_risk_at_coverage_90', 'test_risk_at_coverage_95', 'test_temperature',
                            'error']

def normalize_record(record):
    """Ensure every record has the same column set so incremental CSV append does not misalign columns."""
    for key in STANDARD_RECORD_COLUMNS:
        record.setdefault(key, float('nan'))
    return {key: record.get(key) for key in STANDARD_RECORD_COLUMNS}

def seed_everything(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic

def make_loaders(train_rows, val_rows, test_rows):
    return {
        'train': DataLoader(HistologyDataset(train_rows, True), batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True),
        'val': DataLoader(HistologyDataset(val_rows, False), batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True),
        'test': DataLoader(HistologyDataset(test_rows, False), batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True),
    }

@torch.inference_mode()
def logits_and_labels(model, loader):
    model.eval()
    logits_all, labels_all = [], []
    for x, y, _ in loader:
        logits_all.append(model(x.to(DEVICE, non_blocking=True)).detach().cpu().numpy())
        labels_all.append(np.asarray(y))
    return np.concatenate(logits_all), np.concatenate(labels_all)

def compute_all_metrics(logits, labels, temperature=1.0):
    scaled = logits / temperature
    exp = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    probs = exp / exp.sum(axis=1, keepdims=True)
    predictions = probs.argmax(axis=1)
    metrics = {
        'accuracy': accuracy_score(labels, predictions),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, average='macro'),
        'ece': expected_calibration_error(probs, labels),
        'brier': brier_multiclass(probs, labels),
    }
    try:
        if NUM_CLASSES == 2:
            metrics['auc'] = roc_auc_score(labels, probs[:, 1])
        else:
            metrics['auc'] = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except ValueError:
        metrics['auc'] = float('nan')
    coverage, risk, rcauc = risk_coverage_curve(probs, labels)
    metrics['risk_coverage_auc'] = rcauc
    for target in (0.80, 0.90, 0.95):
        idx = int(target * len(coverage)) - 1
        metrics['risk_at_coverage_' + str(int(100 * target))] = float(risk[idx]) if idx >= 0 else float('nan')
    metrics['temperature'] = float(temperature)
    return metrics, probs

def train_from_scratch(model_name, held_out_mag, patient_fraction, seed):
    seed_everything(seed)
    subset = patient_lomo_split(MANIFEST, held_out_mag, fraction=patient_fraction, seed=seed)
    loaders = make_loaders(subset[subset.split == 'train'], subset[subset.split == 'val'], subset[subset.split == 'test'])
    model = build_from_scratch(model_name, NUM_CLASSES).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()
    best_score = -1.0
    run_id = 'scratch_' + model_name + '_ho' + str(held_out_mag) + '_frac' + str(patient_fraction) + '_seed' + str(seed)
    run_dir = Path(OUTPUT, 'runs', run_id)
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    epochs_log = []
    best_ckpt = run_dir / 'best.pt'
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        for x, y, _ in loaders['train']:
            optim.zero_grad(set_to_none=True)
            loss = loss_fn(model(x.to(DEVICE, non_blocking=True)), y.to(DEVICE, non_blocking=True))
            loss.backward()
            optim.step()
            running += float(loss.detach()) * len(y)
        val_logits, val_labels = logits_and_labels(model, loaders['val'])
        val_metrics, _ = compute_all_metrics(val_logits, val_labels)
        epoch_entry = {'epoch': epoch, 'train_loss': running / len(loaders['train'].dataset)}
        for k, v in val_metrics.items():
            epoch_entry['val_' + k] = v
        epochs_log.append(epoch_entry)
        if val_metrics['macro_f1'] > best_score:
            best_score = val_metrics['macro_f1']
            torch.save(model.state_dict(), best_ckpt)
    model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    val_logits, val_labels = logits_and_labels(model, loaders['val'])
    temperature = fit_temperature(val_logits, val_labels) if TEMPERATURE_SCALE else 1.0
    test_logits, test_labels = logits_and_labels(model, loaders['test'])
    test_metrics, test_probs = compute_all_metrics(test_logits, test_labels, temperature)
    record = normalize_record({
        'run_id': run_id, 'branch': 'from_scratch', 'model': model_name,
        'held_out_mag': held_out_mag, 'patient_fraction': patient_fraction, 'seed': seed,
        'peak_memory_mib': torch.cuda.max_memory_allocated() / 2**20,
        'wallclock_s': time.perf_counter() - start,
        **{'test_' + k: v for k, v in test_metrics.items()},
    })
    pd.DataFrame(epochs_log).to_csv(run_dir / 'epochs.csv', index=False)
    np.savez_compressed(run_dir / 'test_outputs.npz', logits=test_logits, labels=test_labels, probs=test_probs)
    Path(run_dir, 'record.json').write_text(json.dumps(record, indent=2, default=str))
    del model
    torch.cuda.empty_cache()
    return record

def train_fm_linear_probe(held_out_mag, patient_fraction, seed):
    if not USE_FOUNDATION_MODEL:
        return None
    seed_everything(seed)
    subset = patient_lomo_split(MANIFEST, held_out_mag, fraction=patient_fraction, seed=seed)
    loaders = make_loaders(subset[subset.split == 'train'], subset[subset.split == 'val'], subset[subset.split == 'test'])
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    tr_feats, tr_labels, _ = extract_features(FM_MODEL, loaders['train'])
    va_feats, va_labels, _ = extract_features(FM_MODEL, loaders['val'])
    te_feats, te_labels, _ = extract_features(FM_MODEL, loaders['test'])
    probe = LinearProbe(FM_DIM, NUM_CLASSES).to(DEVICE)
    optim = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    best_score = -1.0
    best_state = None
    for epoch in range(200):
        probe.train()
        perm = torch.randperm(len(tr_feats))
        for start_idx in range(0, len(perm), 256):
            idx = perm[start_idx:start_idx + 256]
            xb = tr_feats[idx].to(DEVICE)
            yb = tr_labels[idx].to(DEVICE, dtype=torch.long)
            optim.zero_grad(set_to_none=True)
            loss_fn(probe(xb), yb).backward()
            optim.step()
        probe.eval()
        with torch.inference_mode():
            val_logits = probe(va_feats.to(DEVICE)).cpu().numpy()
        val_metrics, _ = compute_all_metrics(val_logits, va_labels.numpy())
        if val_metrics['macro_f1'] > best_score:
            best_score = val_metrics['macro_f1']
            best_state = {k: v.detach().clone() for k, v in probe.state_dict().items()}
    probe.load_state_dict(best_state)
    probe.eval()
    with torch.inference_mode():
        val_logits = probe(va_feats.to(DEVICE)).cpu().numpy()
        test_logits = probe(te_feats.to(DEVICE)).cpu().numpy()
    temperature = fit_temperature(val_logits, va_labels.numpy()) if TEMPERATURE_SCALE else 1.0
    test_metrics, test_probs = compute_all_metrics(test_logits, te_labels.numpy(), temperature)
    run_id = 'fm_' + FM_NAME.replace('/', '_') + '_ho' + str(held_out_mag) + '_frac' + str(patient_fraction) + '_seed' + str(seed)
    run_dir = Path(OUTPUT, 'runs', run_id)
    run_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(run_dir / 'test_outputs.npz', logits=test_logits, labels=te_labels.numpy(), probs=test_probs)
    # Probe head kept for the BACH OOD cell; saved before record.json marks the run finished.
    torch.save({'state_dict': {k: v.cpu() for k, v in best_state.items()}, 'feature_dim': int(FM_DIM),
                'num_classes': NUM_CLASSES, 'fm_name': FM_NAME, 'temperature': float(temperature)},
               run_dir / 'probe.pt')
    record = normalize_record({
        'run_id': run_id, 'branch': 'fm_linear_probe', 'model': FM_NAME,
        'held_out_mag': held_out_mag, 'patient_fraction': patient_fraction, 'seed': seed,
        'peak_memory_mib': torch.cuda.max_memory_allocated() / 2**20,
        'wallclock_s': time.perf_counter() - start,
        **{'test_' + k: v for k, v in test_metrics.items()},
    })
    Path(run_dir, 'record.json').write_text(json.dumps(record, indent=2, default=str))
    return record

print('Training primitives ready. STANDARD_RECORD_COLUMNS enforced.')


In [ ]:
# MedMamba ablation factory, plus a smoke test and compute report for the ablation task.
# Nothing in this cell trains.
import torch, torch.nn as nn


class NoSSMBlock(nn.Module):
    """Drop-in replacement for SS2D. Same NHWC convention as MedMamba's SS2D.
    Uses LayerNorm -> 3x3 depthwise conv -> 1x1 pointwise -> SiLU.
    Preserves channel dimension so the surrounding VSS block still works."""
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dwconv = nn.Conv2d(d_model, d_model, 3, padding=1, groups=d_model, bias=True)
        self.pwconv = nn.Conv2d(d_model, d_model, 1, bias=True)
        self.act = nn.SiLU()

    def forward(self, x, *args, **kwargs):
        # MedMamba's SS2D always receives (B, H, W, C) — NHWC. Some rare wrappers pass (B, N, C).
        if x.dim() == 4:                                  # (B, H, W, C) NHWC
            y = self.norm(x)                              # LayerNorm on last dim (C)
            y = y.permute(0, 3, 1, 2)                     # -> (B, C, H, W) for Conv2d
            y = self.act(self.pwconv(self.dwconv(y)))
            return y.permute(0, 2, 3, 1)                  # -> NHWC
        if x.dim() == 3:                                  # (B, N, C) sequence form
            B, N, C = x.shape
            HW = int(N ** 0.5)
            y = self.norm(x).transpose(1, 2).reshape(B, C, HW, HW)
            y = self.act(self.pwconv(self.dwconv(y)))
            return y.reshape(B, C, N).transpose(1, 2)
        # NCHW fallback (shouldn't happen in MedMamba, but safe)
        return self.act(self.pwconv(self.dwconv(x)))


def _infer_channels_from_ss2d(mod):
    """Recover the channel dim SS2D was configured with, robust to attribute-name drift."""
    for name in ('d_model', 'd_inner'):
        if hasattr(mod, name):
            v = getattr(mod, name)
            if isinstance(v, int) and v > 0:
                # d_inner is typically 2 * d_model; NoSSMBlock only needs the outer d_model,
                # which SS2D's in_proj / out_proj expose reliably.
                if name == 'd_inner':
                    continue
                return v
    if hasattr(mod, 'in_proj') and hasattr(mod.in_proj, 'in_features'):
        return int(mod.in_proj.in_features)
    if hasattr(mod, 'out_proj') and hasattr(mod.out_proj, 'out_features'):
        return int(mod.out_proj.out_features)
    for p in mod.parameters():
        if p.dim() >= 2:
            return int(p.shape[-1])
    raise RuntimeError('Cannot infer channel dim from SS2D module.')


def _replace_ss2d_in_place(model):
    """Walk model, replace every module whose class name contains 'SS2D' with NoSSMBlock.
    Uses runtime class-name matching so no sys.modules lookup is needed."""
    n_replaced = 0
    for parent in model.modules():
        for child_name, child in list(parent.named_children()):
            if 'SS2D' in child.__class__.__name__:
                d = _infer_channels_from_ss2d(child)
                setattr(parent, child_name, NoSSMBlock(d))
                n_replaced += 1
    return n_replaced


def _medmamba_base(num_classes):
    if ABLATION_BASE_MODEL not in EXTERNAL_FACTORIES:
        raise RuntimeError('MedMamba is not registered; the sources cell (c3) must succeed first.')
    return EXTERNAL_FACTORIES[ABLATION_BASE_MODEL](num_classes)


def build_medmamba_no_ssm(num_classes):
    """Every SS2D branch of MedMamba's SS_Conv_SSM blocks becomes NoSSMBlock (depthwise conv)."""
    model = _medmamba_base(num_classes)
    n = _replace_ss2d_in_place(model)
    if n == 0:
        raise RuntimeError('No SS2D modules found in MedMamba; the no-SSM ablation would be a no-op.')
    print(f'  medmamba_no_ssm: replaced {n} SS2D modules with NoSSMBlock')
    return model


ABLATION_FACTORIES = {
    'medmamba_no_ssm': build_medmamba_no_ssm,
}
assert set(ABLATION_FACTORIES) == set(ABLATION_VARIANTS), 'ABLATION_VARIANTS and the factories disagree'
for _name, _fn in ABLATION_FACTORIES.items():
    EXTERNAL_FACTORIES[_name] = _fn
print('Registered ablation factories:', list(ABLATION_FACTORIES))


if PARTC_TASK == 'ablation':
    print('\nSmoke test + compute report (last check before training):')
    _rows = []
    for _name, _fn in [(ABLATION_BASE_MODEL, _medmamba_base)] + list(ABLATION_FACTORIES.items()):
        try:
            m = _fn(NUM_CLASSES).to(DEVICE).eval()
            n_params = sum(p.numel() for p in m.parameters())
            _dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
            with torch.no_grad():
                out = m(_dummy)
            assert tuple(out.shape) == (1, NUM_CLASSES), f'output shape {tuple(out.shape)}'
            try:
                from fvcore.nn import FlopCountAnalysis
                _flops = FlopCountAnalysis(m, _dummy).total() / 1e9
            except Exception:
                _flops = float('nan')
            _rows.append({'model': _name, 'parameters': int(n_params), 'flops_g_at_224': _flops})
            _fl = f'{_flops:.2f}' if _flops == _flops else 'n/a'
            print(f'  {_name:<22} OK  params={n_params/1e6:.2f}M  FLOPs={_fl}G')
            del m, _dummy
            torch.cuda.empty_cache()
        except Exception as e:
            _rows.append({'model': _name, 'parameters': -1, 'flops_g_at_224': float('nan')})
            print(f'  {_name:<22} BUILD FAILED: {repr(e)[:220]}')
            print('    -> this variant will fail when the training loop reaches it.')
    pd.DataFrame(_rows).to_csv(Path(OUTPUT, 'ablation_compute_report.csv'), index=False)
    print(f'Wrote {OUTPUT}/ablation_compute_report.csv')


In [ ]:
# Patient-level k-fold split at ho=400.
# Uses the SAME held-out-magnification LOMO scheme, but partitions PATIENTS into k folds
# (stratified by class) instead of a single val/test split. Each fold rotates: 1/k patients
# form the test set, 1/k for val, remainder for train (both classes represented in every split).
def patient_kfold_lomo_split(frame, held_out_mag, k, fold_idx, seed,
                              val_ratio=0.15, fraction=1.0):
    """Return the same DataFrame with a 'split' column, using fold_idx (0..k-1).

    Sized to match patient_lomo_split EXACTLY so fold means are directly comparable
    to the LOMO baseline:
        test  = this fold's patients @ held_out_mag            -> 1/k  (k=5 => 20%, = test_ratio)
        val   = val_ratio of ALL patients of the class         -> 15%  (= val_ratio)
        train = the remainder                                  -> 65%
    Taking val as a fraction of the *remaining* patients instead would give
    0.15 * 0.80 = 12% val and 68% train, i.e. 3% more training data per fold than the
    baseline -- which would inflate the CV mean and invalidate the comparison.

    The val window rotates with fold_idx so validation patients differ across folds.

    Class balance is enforced in every split."""
    assert 0 <= fold_idx < k, f'fold_idx {fold_idx} out of range for k={k}'
    frame = frame.copy()
    patient_class = frame.groupby('patient_id').coarse_label.first().reset_index()
    rng = np.random.default_rng(seed)

    fold_test = {}
    fold_val  = {}
    fold_train = {}
    for cls in sorted(patient_class.coarse_label.unique()):
        group = patient_class.loc[patient_class.coarse_label == cls, 'patient_id'].to_numpy()
        perm = rng.permutation(group)
        # Stratified k-fold: split perm into k roughly-equal buckets.
        buckets = np.array_split(perm, k)
        test = list(buckets[fold_idx])
        remaining = np.concatenate([buckets[i] for i in range(k) if i != fold_idx])
        # From remaining, carve val (stratified stays: still class-cls only).
        # val is a fraction of ALL patients in this class (not of `remaining`), so that
        # train/val/test proportions equal patient_lomo_split's 65/15/20.
        n_val = max(1, int(round(val_ratio * len(perm))))
        assert n_val < len(remaining), (
            f'class {cls}: val size {n_val} >= remaining {len(remaining)}; '
            f'reduce val_ratio or k')
        # Rotate the val window with fold_idx. `remaining` concatenates the non-test
        # buckets in index order, so a fixed remaining[:n_val] would hand folds 1..4 the
        # same leading patients from bucket[0] and never train on them.
        span = max(1, len(remaining) - n_val)
        val_start = (fold_idx * n_val) % span
        val = list(remaining[val_start:val_start + n_val])
        train_pool = list(remaining[:val_start]) + list(remaining[val_start + n_val:])
        # Optional patient-fraction: subsample train patients only.
        n_keep = max(1, int(round(fraction * len(train_pool))))
        train = train_pool[:n_keep]
        fold_test.setdefault(cls, []).extend(test)
        fold_val.setdefault(cls,  []).extend(val)
        fold_train.setdefault(cls,[]).extend(train)

    test_p  = set(sum(fold_test.values(), []))
    val_p   = set(sum(fold_val.values(),  []))
    train_p = set(sum(fold_train.values(),[]))

    train_mags = [m for m in MAGNIFICATIONS if m != held_out_mag]
    frame['split'] = 'drop'
    frame.loc[frame.patient_id.isin(train_p) & frame.magnification.isin(train_mags), 'split'] = 'train'
    frame.loc[frame.patient_id.isin(val_p)   & frame.magnification.isin(train_mags), 'split'] = 'val'
    frame.loc[frame.patient_id.isin(test_p)  & (frame.magnification == held_out_mag), 'split'] = 'test'
    subset = frame.loc[frame.split != 'drop'].copy()

    # Sanity checks
    leak = subset.groupby('patient_id').split.nunique().gt(1).any()
    assert not leak, f'Patient leakage in fold {fold_idx}.'
    for name in ('train', 'val', 'test'):
        rows = subset[subset.split == name]
        if len(rows) == 0:
            raise RuntimeError(f'Fold {fold_idx} split {name} empty (held_out={held_out_mag}).')
        counts = rows.coarse_label.value_counts().to_dict()
        if len(counts) < 2:
            raise RuntimeError(f'Fold {fold_idx} split {name} has only one class {counts}.')
    return subset

# Probe: show every fold's split summary at ho=400.
print(f'{CV_K}-fold patient-level split preview at held_out_mag={CV_HELD_OUT}:')
for fold_idx in range(CV_K):
    subset = patient_kfold_lomo_split(MANIFEST, CV_HELD_OUT, CV_K, fold_idx, CV_SEED)
    n_pat = subset.groupby('split').patient_id.nunique().to_dict()
    n_img = subset.groupby('split').size().to_dict()
    cls   = {s: subset[subset.split==s].coarse_label.value_counts().to_dict() for s in ('train','val','test')}
    total_pat = sum(n_pat.values())
    pct = {s: f'{100*n_pat[s]/total_pat:.0f}%' for s in n_pat}
    print(f'  fold {fold_idx}: patients={n_pat} {pct} images={n_img} classes={cls}')

# Compare against the single-split baseline so the proportions can be eyeballed.
_ref = patient_lomo_split(MANIFEST, CV_HELD_OUT, fraction=1.0, seed=CV_SEED)
_ref_pat = _ref.groupby('split').patient_id.nunique().to_dict()
_ref_total = sum(_ref_pat.values())
print(f'  baseline patient_lomo_split for reference: {_ref_pat} '
      f'{{{", ".join(f"{s}: {100*_ref_pat[s]/_ref_total:.0f}%" for s in _ref_pat)}}}')
print('  -> fold train/val/test percentages should match the baseline row above.')


In [ ]:
# Part C protocol for this account's PARTC_TASK: the MedMamba ablation, or cross-validation.
# Resumable (runs with a record.json are skipped), incremental (raw_runs.csv is appended after
# every run) and time-budgeted (a run starts only if it should finish in time; see the time guard).
raw_runs_csv = Path(OUTPUT, 'raw_runs.csv')


def _run_exists(run_id):
    return Path(OUTPUT, 'runs', run_id, 'record.json').exists()

def _append_row(record):
    record = normalize_record(dict(record))
    row = pd.DataFrame([record], columns=STANDARD_RECORD_COLUMNS)
    header = not raw_runs_csv.exists()
    row.to_csv(raw_runs_csv, mode='a', header=header, index=False)

def train_variant(model_name, held_out_mag, patient_fraction, seed, run_prefix, branch):
    """Single code path for BOTH the baseline reference and the ablated variants.

    Identical to train_from_scratch except for the run_id prefix and the recorded
    branch. Using one function guarantees the baseline and every ablation see the
    same split, same optimizer, same epoch count and same checkpoint rule, so the
    reported delta is attributable to the architecture change alone."""
    seed_everything(seed)
    subset = patient_lomo_split(MANIFEST, held_out_mag, fraction=patient_fraction, seed=seed)
    loaders = make_loaders(subset[subset.split == 'train'], subset[subset.split == 'val'], subset[subset.split == 'test'])
    model = build_from_scratch(model_name, NUM_CLASSES).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()
    best_score = -1.0
    run_id = f'{run_prefix}_{model_name}_ho{held_out_mag}_frac{patient_fraction}_seed{seed}'
    run_dir = Path(OUTPUT, 'runs', run_id)
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    epochs_log = []
    best_ckpt = run_dir / 'best.pt'
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        for x, y, _ in loaders['train']:
            optim.zero_grad(set_to_none=True)
            loss = loss_fn(model(x.to(DEVICE, non_blocking=True)), y.to(DEVICE, non_blocking=True))
            loss.backward(); optim.step()
            running += float(loss.detach()) * len(y)
        val_logits, val_labels = logits_and_labels(model, loaders['val'])
        val_metrics, _ = compute_all_metrics(val_logits, val_labels)
        entry = {'epoch': epoch, 'train_loss': running / len(loaders['train'].dataset)}
        for k, v in val_metrics.items(): entry['val_' + k] = v
        epochs_log.append(entry)
        if val_metrics['macro_f1'] > best_score:
            best_score = val_metrics['macro_f1']
            torch.save(model.state_dict(), best_ckpt)
    model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    val_logits, val_labels = logits_and_labels(model, loaders['val'])
    temperature = fit_temperature(val_logits, val_labels) if TEMPERATURE_SCALE else 1.0
    test_logits, test_labels = logits_and_labels(model, loaders['test'])
    test_metrics, test_probs = compute_all_metrics(test_logits, test_labels, temperature)
    record = normalize_record({
        'run_id': run_id, 'branch': branch, 'model': model_name,
        'held_out_mag': held_out_mag, 'patient_fraction': patient_fraction, 'seed': seed,
        'peak_memory_mib': torch.cuda.max_memory_allocated() / 2**20,
        'wallclock_s': time.perf_counter() - start,
        **{'test_' + k: v for k, v in test_metrics.items()},
    })
    pd.DataFrame(epochs_log).to_csv(run_dir / 'epochs.csv', index=False)
    np.savez_compressed(run_dir / 'test_outputs.npz', logits=test_logits, labels=test_labels, probs=test_probs)
    Path(run_dir, 'record.json').write_text(json.dumps(record, indent=2, default=str))
    del model; torch.cuda.empty_cache()
    return record

def train_cv_fold(model_name, held_out_mag, patient_fraction, seed, fold_idx):
    """5-fold CV variant: uses patient_kfold_lomo_split instead of the single-split version."""
    seed_everything(seed)
    subset = patient_kfold_lomo_split(MANIFEST, held_out_mag, CV_K, fold_idx, seed,
                                        fraction=patient_fraction)
    loaders = make_loaders(subset[subset.split == 'train'], subset[subset.split == 'val'], subset[subset.split == 'test'])
    model = build_from_scratch(model_name, NUM_CLASSES).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()
    best_score = -1.0
    run_id = f'cv_{model_name}_ho{held_out_mag}_frac{patient_fraction}_seed{seed}_fold{fold_idx}'
    run_dir = Path(OUTPUT, 'runs', run_id)
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    best_ckpt = run_dir / 'best.pt'
    epochs_log = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        for x, y, _ in loaders['train']:
            optim.zero_grad(set_to_none=True)
            loss = loss_fn(model(x.to(DEVICE, non_blocking=True)), y.to(DEVICE, non_blocking=True))
            loss.backward(); optim.step()
            running += float(loss.detach()) * len(y)
        val_logits, val_labels = logits_and_labels(model, loaders['val'])
        val_metrics, _ = compute_all_metrics(val_logits, val_labels)
        entry = {'epoch': epoch, 'train_loss': running / len(loaders['train'].dataset)}
        for k, v in val_metrics.items(): entry['val_' + k] = v
        epochs_log.append(entry)
        if val_metrics['macro_f1'] > best_score:
            best_score = val_metrics['macro_f1']
            torch.save(model.state_dict(), best_ckpt)
    model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    val_logits, val_labels = logits_and_labels(model, loaders['val'])
    temperature = fit_temperature(val_logits, val_labels) if TEMPERATURE_SCALE else 1.0
    test_logits, test_labels = logits_and_labels(model, loaders['test'])
    test_metrics, test_probs = compute_all_metrics(test_logits, test_labels, temperature)
    record = normalize_record({
        'run_id': run_id, 'branch': 'cv', 'model': model_name,
        'held_out_mag': held_out_mag, 'patient_fraction': patient_fraction, 'seed': seed,
        'peak_memory_mib': torch.cuda.max_memory_allocated() / 2**20,
        'wallclock_s': time.perf_counter() - start,
        **{'test_' + k: v for k, v in test_metrics.items()},
    })
    record['fold_idx'] = fold_idx        # Extra column for CV; ignored by analysis if absent.
    pd.DataFrame(epochs_log).to_csv(run_dir / 'epochs.csv', index=False)
    np.savez_compressed(run_dir / 'test_outputs.npz', logits=test_logits, labels=test_labels, probs=test_probs)
    Path(run_dir, 'record.json').write_text(json.dumps(record, indent=2, default=str))
    del model; torch.cuda.empty_cache()
    return record


def train_cv_fold_fm(held_out_mag, patient_fraction, seed, fold_idx):
    """Frozen-FM linear probe on one CV fold: train_fm_linear_probe's recipe with the split
    swapped for patient_kfold_lomo_split."""
    if not USE_FOUNDATION_MODEL or FM_MODEL is None:
        raise RuntimeError('Foundation model not loaded; the CV task needs the FM cell to succeed.')
    seed_everything(seed)
    subset = patient_kfold_lomo_split(MANIFEST, held_out_mag, CV_K, fold_idx, seed,
                                      fraction=patient_fraction)
    loaders = make_loaders(subset[subset.split == 'train'], subset[subset.split == 'val'], subset[subset.split == 'test'])
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    tr_feats, tr_labels, _ = extract_features(FM_MODEL, loaders['train'])
    va_feats, va_labels, _ = extract_features(FM_MODEL, loaders['val'])
    te_feats, te_labels, _ = extract_features(FM_MODEL, loaders['test'])
    probe = LinearProbe(FM_DIM, NUM_CLASSES).to(DEVICE)
    optim = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    best_score = -1.0
    best_state = None
    for epoch in range(200):
        probe.train()
        perm = torch.randperm(len(tr_feats))
        for start_idx in range(0, len(perm), 256):
            idx = perm[start_idx:start_idx + 256]
            xb = tr_feats[idx].to(DEVICE)
            yb = tr_labels[idx].to(DEVICE, dtype=torch.long)
            optim.zero_grad(set_to_none=True)
            loss_fn(probe(xb), yb).backward()
            optim.step()
        probe.eval()
        with torch.inference_mode():
            val_logits = probe(va_feats.to(DEVICE)).cpu().numpy()
        val_metrics, _ = compute_all_metrics(val_logits, va_labels.numpy())
        if val_metrics['macro_f1'] > best_score:
            best_score = val_metrics['macro_f1']
            best_state = {k: v.detach().clone() for k, v in probe.state_dict().items()}
    probe.load_state_dict(best_state)
    probe.eval()
    with torch.inference_mode():
        val_logits = probe(va_feats.to(DEVICE)).cpu().numpy()
        test_logits = probe(te_feats.to(DEVICE)).cpu().numpy()
    temperature = fit_temperature(val_logits, va_labels.numpy()) if TEMPERATURE_SCALE else 1.0
    test_metrics, test_probs = compute_all_metrics(test_logits, te_labels.numpy(), temperature)
    run_id = f"cv_{FM_NAME.replace('/', '_')}_ho{held_out_mag}_frac{patient_fraction}_seed{seed}_fold{fold_idx}"
    run_dir = Path(OUTPUT, 'runs', run_id)
    run_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(run_dir / 'test_outputs.npz', logits=test_logits, labels=te_labels.numpy(), probs=test_probs)
    record = normalize_record({
        'run_id': run_id, 'branch': 'cv', 'model': FM_NAME,
        'held_out_mag': held_out_mag, 'patient_fraction': patient_fraction, 'seed': seed,
        'peak_memory_mib': torch.cuda.max_memory_allocated() / 2**20,
        'wallclock_s': time.perf_counter() - start,
        **{'test_' + k: v for k, v in test_metrics.items()},
    })
    record['fold_idx'] = fold_idx
    Path(run_dir, 'record.json').write_text(json.dumps(record, indent=2, default=str))
    return record


# ---------- Queue ----------
planned = []
if PARTC_TASK == 'ablation':
    if INCLUDE_BASELINE_REFERENCE:
        for seed in BASELINE_SEEDS:
            rid = f'baselineref_{BASELINE_MODEL}_ho{ABLATION_HELD_OUT}_frac{ABLATION_PATIENT_FRACTION}_seed{seed}'
            planned.append(('baseline_ref', BASELINE_MODEL, ABLATION_HELD_OUT,
                            ABLATION_PATIENT_FRACTION, seed, None, rid))
    for variant in ABLATION_VARIANTS:
        for seed in ABLATION_SEEDS:
            rid = f'ablation_{variant}_ho{ABLATION_HELD_OUT}_frac{ABLATION_PATIENT_FRACTION}_seed{seed}'
            planned.append(('ablation', variant, ABLATION_HELD_OUT, ABLATION_PATIENT_FRACTION, seed, None, rid))
else:
    # The minute-long Phikon folds go first: if the session is cut short, the check on the
    # headline claims is already complete.
    for model_name, mags, folds in sorted(CV_PLAN, key=lambda plan: plan[0] != FM_NAME):
        tag = model_name.replace('/', '_')
        for ho in mags:
            for fold_idx in folds:
                rid = f'cv_{tag}_ho{ho}_frac{CV_PATIENT_FRACTION}_seed{CV_SEED}_fold{fold_idx}'
                planned.append(('cv', model_name, ho, CV_PATIENT_FRACTION, CV_SEED, fold_idx, rid))

print(f'Planned runs for PARTC_TASK={PARTC_TASK!r}: {len(planned)}')
completed = [e for e in planned if _run_exists(e[-1])]
remaining = [e for e in planned if not _run_exists(e[-1])]
print(f'Already completed on disk: {len(completed)}   Remaining: {len(remaining)}')

if completed and raw_runs_csv.exists():
    raw_runs_csv.unlink()
for entry in completed:
    _append_row(json.loads(Path(OUTPUT, 'runs', entry[-1], 'record.json').read_text()))

# ---- Time guard ----
# Kaggle stops a commit 12 h after its kernel starts, setup cells included, and nothing here
# interrupts a run already under way. So a run is started only if it should finish within
# MAX_RUNTIME_HOURS of kernel start, leaving POST_LOOP_RESERVE_H for the BACH and aggregate
# cells. Expected length = 1.1x the longest finished run of the same model (this commit or
# resumed records), else DEFAULT_RUN_HOURS. A run that does not fit is left for the next
# commit; shorter runs further down the queue still get their chance.
POST_LOOP_RESERVE_H = 0.5
DEFAULT_RUN_HOURS = {'medmamba': 1.25, 'medmamba_no_ssm': 1.25, FM_NAME: 0.1}
try:
    import psutil
    KERNEL_START = psutil.Process(os.getpid()).create_time()
except Exception:
    KERNEL_START = time.time()
    print('psutil unavailable: the time budget counts from this cell, not from kernel start.')
deadline = KERNEL_START + (MAX_RUNTIME_HOURS - POST_LOOP_RESERVE_H) * 3600
print(f'Setup cells took {(time.time() - KERNEL_START) / 60:.1f} min; '
      f'runs must finish within the next {(deadline - time.time()) / 3600:.2f} h.')

longest_run_s = {}
def _note_duration(record):
    seconds = (record or {}).get('wallclock_s')
    if isinstance(seconds, (int, float)) and seconds == seconds and seconds > 0:
        longest_run_s[record['model']] = max(longest_run_s.get(record['model'], 0.0), float(seconds))

def _expected_run_s(model_name):
    if model_name in longest_run_s:
        return 1.1 * longest_run_s[model_name]
    return DEFAULT_RUN_HOURS.get(model_name, 1.75) * 3600

def _fits(model_name):
    return time.time() + _expected_run_s(model_name) <= deadline

for entry in completed:
    _note_duration(json.loads(Path(OUTPUT, 'runs', entry[-1], 'record.json').read_text()))

session_start = time.perf_counter()
left_for_next_commit = []

for idx, (branch, model_name, held_out, fraction, seed, fold_idx, run_id) in enumerate(remaining, 1):
    elapsed = time.perf_counter() - session_start
    if not _fits(model_name):
        left_for_next_commit.append(run_id)
        print(f'[{idx}/{len(remaining)}] not started: {run_id} needs ~{_expected_run_s(model_name)/3600:.2f} h, '
              f'{max(deadline - time.time(), 0)/3600:.2f} h left')
        continue
    tag = f' fold={fold_idx}' if fold_idx is not None else ''
    print(f'[{idx}/{len(remaining)}] elapsed {elapsed/3600:.2f} h  {branch}  {model_name}  '
          f'ho={held_out} seed={seed}{tag}')
    try:
        if branch in ('ablation', 'baseline_ref'):
            prefix = 'ablation' if branch == 'ablation' else 'baselineref'
            rec = train_variant(model_name, held_out, fraction, seed, prefix, branch)
        elif model_name == FM_NAME:
            rec = train_cv_fold_fm(held_out, fraction, seed, fold_idx)
        else:
            rec = train_cv_fold(model_name, held_out, fraction, seed, fold_idx)
        _append_row(rec)
        _note_duration(rec)
    except Exception as exc:
        _append_row({'run_id': run_id, 'branch': branch, 'model': model_name,
                     'held_out_mag': held_out, 'patient_fraction': fraction, 'seed': seed,
                     'error': repr(exc)[:400]})
        print('  -> FAILED:', repr(exc)[:200])

print(f'Session finished. Total wall-clock: {(time.perf_counter() - session_start)/3600:.2f} h')
if left_for_next_commit:
    print(f'Time budget reached: {len(left_for_next_commit)} run(s) left for the next commit.')
    print('Attach this version as input and Save & Run All again (see the resume cell).')

# ---------- Session summary ----------
if raw_runs_csv.exists():
    _done = pd.read_csv(raw_runs_csv)
    if 'error' in _done.columns:
        _failed = _done[_done.error.notna()]
        if len(_failed):
            print(f'\n{len(_failed)} run(s) FAILED:')
            for _, r in _failed.iterrows():
                print(f'  {r.run_id}: {str(r.error)[:160]}')
        _done = _done[_done.error.isna()]
    print('\n' + '=' * 72)
    if PARTC_TASK == 'ablation':
        print(f'ABLATION @ ho={ABLATION_HELD_OUT}  (macro-F1, mean over seeds)')
        _base = _done[_done.branch == 'baseline_ref']
        _b = _base.test_macro_f1.mean() if len(_base) else None
        if _b is not None:
            print(f'  {BASELINE_MODEL + " (baseline)":<26} {_b:.4f}  (n={len(_base)})')
        for v in ABLATION_VARIANTS:
            sub = _done[(_done.branch == 'ablation') & (_done.model == v)]
            if len(sub):
                d = f'  delta={sub.test_macro_f1.mean() - _b:+.4f}' if _b is not None else ''
                print(f'  {v:<26} {sub.test_macro_f1.mean():.4f}{d}  (n={len(sub)})')
        if _b is None:
            print('  Deltas against the MedMamba ho=400 LOMO runs are computed in the analysis notebook (Fig 13, T9).')
    else:
        print(f'{CV_K}-FOLD CV  (mean +/- SD over folds)')
        for (m, ho), sub in _done[_done.branch == 'cv'].groupby(['model', 'held_out_mag']):
            parts = '  '.join(f'{lbl}={sub[c].mean():.4f}+/-{sub[c].std(ddof=1):.4f}'
                              for lbl, c in (('F1', 'test_macro_f1'), ('BAcc', 'test_balanced_accuracy'),
                                             ('AUC', 'test_auc'))
                              if c in sub.columns)
            print(f'  {m:<14} ho={int(ho):<4} n={len(sub)}  {parts}')
    print('=' * 72)


In [ ]:
# ---------- BACH as an OOD probe for the Part C checkpoints ----------
# Inference only: every checkpoint written by the protocol above is run over the BACH
# images and scored with MSP and energy. A few seconds per checkpoint, no training.
#
# It selects branch in PARTC_OOD_BRANCHES rather than Part A/B's 'from_scratch' and
# 'fm_linear_probe', because none of Part C's runs carry those branches. Like Part A/B it
# writes a `branch` column: the analysis notebook concatenates every bach_ood_scores.csv
# and splits LOMO rows from Part C rows on that column, so MedMamba's OOD number at
# ho=400 does not blend the LOMO runs with the ablation runs.
# CV folds are not scored: Fig 13's OOD panel needs only the ablation variants.
PARTC_OOD_BRANCHES = ('baseline_ref', 'ablation')

def discover_bach_root(hint=None, search_root='/kaggle/input'):
    candidates = []
    if hint and Path(hint).exists():
        candidates.append(Path(hint))
    if Path(search_root).exists():
        for root, dirs, _files in os.walk(search_root):
            if {'Normal', 'Benign', 'InSitu', 'Invasive'}.issubset(set(dirs)):
                candidates.append(Path(root))
    best = None; best_count = 0
    for candidate in set(candidates):
        count = (sum(1 for _ in Path(candidate).rglob('*.tif'))
                 + sum(1 for _ in Path(candidate).rglob('*.png')))
        if count > best_count:
            best = candidate; best_count = count
    if best is None:
        print('BACH root not found. OOD pass will be skipped.')
        print('  To enable it: Add Input > truthisneverlinear/bach-breast-cancer-histology-images')
        return None
    print(f'Auto-discovered BACH_ROOT = {best}  ({best_count} image files)')
    return str(best)

if BACH_ROOT is None:
    BACH_ROOT = discover_bach_root()
_raw_runs_csv = Path(OUTPUT, 'raw_runs.csv')

if BACH_ROOT is None or not _raw_runs_csv.exists():
    print('BACH OOD skipped (no BACH root, or raw_runs.csv missing).')
else:
    try:
        def build_bach_manifest(root):
            rows = []
            for cls, cls_id in {'Normal': 0, 'Benign': 1, 'InSitu': 2, 'Invasive': 3}.items():
                for suffix in ('*.tif', '*.png', '*.jpg'):
                    for path in sorted(Path(root, cls).glob(suffix)):
                        rows.append({'path': str(path), 'bach_class': cls_id})
            return pd.DataFrame(rows)

        bach_frame = build_bach_manifest(BACH_ROOT)
        if bach_frame.empty:
            print('BACH manifest empty at', BACH_ROOT)
        else:
            print(f'BACH manifest: {len(bach_frame)} images '
                  f'{bach_frame.bach_class.value_counts().sort_index().to_dict()}')

            class BachDataset(Dataset):
                def __init__(self, rows):
                    self.rows = rows.reset_index(drop=True)
                    self.transform = transforms.Compose([
                        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        transforms.ToTensor(),
                        transforms.Normalize((.485, .456, .406), (.229, .224, .225))])
                def __len__(self):
                    return len(self.rows)
                def __getitem__(self, index):
                    row = self.rows.iloc[index]
                    return self.transform(Image.open(row.path).convert('RGB')), int(row.bach_class), 'bach'

            bach_loader = DataLoader(BachDataset(bach_frame), batch_size=BATCH_SIZE,
                                     shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

            raw_frame = pd.read_csv(_raw_runs_csv)
            sel = raw_frame[raw_frame.get('branch').isin(PARTC_OOD_BRANCHES)]
            if 'error' in sel.columns:
                sel = sel[sel.error.isna()]
            print(f'Scoring {len(sel)} Part C checkpoints on BACH '
                  f'(branches {list(PARTC_OOD_BRANCHES)})')

            ood_rows = []
            for i, (_, record) in enumerate(sel.iterrows(), 1):
                ckpt_path = Path(OUTPUT, 'runs', record['run_id'], 'best.pt')
                if not ckpt_path.exists():
                    print(f'  [{i}/{len(sel)}] {record["run_id"]}: no best.pt, skipped')
                    continue
                try:
                    model = build_from_scratch(record['model'], NUM_CLASSES).to(DEVICE)
                    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
                    model.eval()
                    bach_logits, _ = logits_and_labels(model, bach_loader)
                    exp = np.exp(bach_logits - bach_logits.max(axis=1, keepdims=True))
                    bach_probs = exp / exp.sum(axis=1, keepdims=True)
                    msp = maximum_softmax_probability(bach_probs)
                    eng = energy_score(bach_logits)
                    ood_rows.append({'run_id': record['run_id'],
                                     'branch': record['branch'],
                                     'model': record['model'],
                                     'held_out_mag': record['held_out_mag'],
                                     'seed': record.get('seed'),
                                     'msp_mean': float(msp.mean()), 'msp_std': float(msp.std()),
                                     'energy_mean': float(eng.mean()), 'energy_std': float(eng.std())})
                    print(f'  [{i}/{len(sel)}] {record["run_id"]}: '
                          f'MSP={msp.mean():.4f}  energy={eng.mean():.3f}')
                    del model
                    torch.cuda.empty_cache()
                except Exception as exc:
                    print(f'  [{i}/{len(sel)}] {record["run_id"]}: FAILED {repr(exc)[:160]}')

            if ood_rows:
                pd.DataFrame(ood_rows).to_csv(Path(OUTPUT, 'bach_ood_scores.csv'), index=False)
                print(f'\nbach_ood_scores.csv written: {len(ood_rows)} rows.')
                _o = pd.DataFrame(ood_rows)
                print('\nMean energy by model (higher magnitude = more OOD-like):')
                for m, v in _o.groupby('model').energy_mean.mean().sort_values().items():
                    print(f'  {m:<24} {v:.3f}')
            else:
                print('\nNo checkpoints scored; bach_ood_scores.csv not written.')
    except Exception as exc:
        print(f'BACH OOD stage skipped: {repr(exc)[:200]}')


In [ ]:
# Aggregation. Folder-only mode by default -- no ZIP (saves ~2x space on Kaggle output).
# Set CREATE_ZIP = True below if you want the .zip archive too (roughly doubles the total).
CREATE_ZIP = False

raw_runs_csv = Path(OUTPUT, 'raw_runs.csv')
raw_frame = pd.read_csv(raw_runs_csv) if raw_runs_csv.exists() else pd.DataFrame()
if not raw_frame.empty:
    ok = raw_frame[raw_frame.error.isna()] if 'error' in raw_frame.columns else raw_frame
    metric_cols = [c for c in ok.columns if c.startswith('test_')]
    if metric_cols:
        summary = (ok.groupby(['branch', 'model', 'held_out_mag', 'patient_fraction'])[metric_cols]
                     .agg(['mean', 'std', 'count']))
        summary.to_csv(Path(OUTPUT, 'five_seed_summary.csv'))
    compute = pd.read_csv(Path(OUTPUT, 'from_scratch_compute_report.csv'))
    if 'branch' in ok.columns:
        from_scratch_wide = ok[ok.branch == 'from_scratch'].merge(compute, on='model', how='left')
    else:
        from_scratch_wide = ok
    from_scratch_wide.to_csv(Path(OUTPUT, 'pareto_scratch.csv'), index=False)

print('Artifacts under', OUTPUT, ':')
total_bytes = 0
for name in ['config.json', 'environment.json', 'source_manifest.json', 'breakhis_manifest.csv',
             'breakhis_manifest.sha256', 'from_scratch_compute_report.csv', 'pilot_records.csv',
             'raw_runs.csv', 'five_seed_summary.csv', 'pareto_scratch.csv', 'bach_ood_scores.csv']:
    path = Path(OUTPUT, name)
    if path.exists():
        sz = path.stat().st_size
        total_bytes += sz
        print('  -', name.ljust(40), str(sz).rjust(12), 'bytes')

runs_dir = Path(OUTPUT, 'runs')
if runs_dir.exists():
    run_folders = sorted([p for p in runs_dir.iterdir() if p.is_dir()])
    per_run_bytes = 0
    for rf in run_folders:
        per_run_bytes += sum(f.stat().st_size for f in rf.rglob('*') if f.is_file())
    print('  runs/', len(run_folders), 'runs,', round(per_run_bytes / 1e9, 2), 'GB (best.pt + epochs.csv + test_outputs.npz + record.json per run)')
    total_bytes += per_run_bytes

print('Total output size:', round(total_bytes / 1e9, 2), 'GB')

if CREATE_ZIP:
    print('Creating ZIP archive...')
    archive = shutil.make_archive('/kaggle/working/' + RUN_NAME, 'zip', OUTPUT)
    print('Archive:', archive)
else:
    print('CREATE_ZIP=False -> no zip written. Analysis notebook can read', OUTPUT, 'directly by attaching this notebook output as input.')
    print('To make a zip anyway, set CREATE_ZIP=True and rerun this cell.')
